In [19]:
import mysql.connector
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score, classification_report
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder


def get_connection():
  return mysql.connector.connect(
      host="localhost",
      user="root",
      password="",
      database="marketdata",
  )


# ------------------------------------------
# 1. FETCH DATA FROM MYSQL WITH CONSTRAINT
# ------------------------------------------
try:
  db = get_connection()
  # Fetch only rows where edited equals "1"
  query = 'SELECT * FROM `labeled_flatdata` WHERE `edited` = "1"'
  df = pd.read_sql(query, db)
  print(
      f"Successfully loaded {len(df)} filtered rows (edited = '1') from MySQL"
      " database."
  )
except Exception as e:
  print(f"Database connection or query failed: {e}")
  df = pd.DataFrame()
finally:
  if "db" in locals() and db.is_connected():
    db.close()


# ------------------------------------------
# 2. PROCEED ONLY IF DATA IS LOADED
# ------------------------------------------
if df.empty:
  print(
      "DataFrame is empty. Check your database connection, credentials, and"
      " whether any rows have edited = '1'."
  )
else:
  # Define all possible target columns
  annotation_columns = [
      "was",
      "then",
      "currently",
      "future",
      "volatility",
      "bias",
      "phase",
      "momentum_direction",
      "momentum_strength",
      "setup",
      "momentum_candle",
      "decision",
  ]

  # Dynamically keep only targets that actually exist in the loaded DataFrame
  valid_targets = [col for col in annotation_columns if col in df.columns]
  print(f"Valid target columns found: {valid_targets}\n")

  # ------------------------------------------
  # 3. INPUT FEATURES: M1 PRICE COLUMNS
  # ------------------------------------------
  feature_columns = [
      col
      for col in df.columns
      if col.startswith("M1_")
      and not col.endswith("_time")
      and col.rsplit("_", 1)[-1] in {"open", "high", "low", "close"}
  ]

  X = df[feature_columns].copy()

  # Convert features to numeric values and handle infinite/missing values
  X = X.apply(pd.to_numeric, errors="coerce")
  X = X.replace([np.inf, -np.inf], np.nan)

  # ------------------------------------------
  # 4. TRAIN A MODEL FOR EACH TARGET
  # ------------------------------------------
  models = {}
  label_encoders = {}
  results = {}

  for target in valid_targets:
    # Robust cleaning: handle missing values, empty strings, and stringified nulls
    y_series = df[target].fillna("").astype(str).str.strip()
    valid = (
        df[target].notna()
        & y_series.ne("")
        & y_series.ne("nan")
        & y_series.ne("None")
        & y_series.ne("NULL")
    )

    X_target = X.loc[valid]
    y_raw = y_series.loc[valid]

    print(f"Target '{target}': found {len(y_raw)} valid labeled rows.")

    # Threshold check (set to 2 for testing, increase as dataset grows)
    if len(y_raw) < 2:
      print(
          f"Skipping {target}: insufficient labeled rows ({len(y_raw)} found,"
          " need at least 2)."
      )
      continue

    # Encode categorical labels as integers
    encoder = LabelEncoder()
    y = encoder.fit_transform(y_raw)

    if len(encoder.classes_) < 2:
      print(
          f"Skipping {target}: only one unique label found ({encoder.classes_}).\n"
      )
      continue

    # Stratification requires enough examples per class
    class_counts = pd.Series(y).value_counts()
    stratify = y if class_counts.min() >= 2 else None
    test_size = 0.2 if len(X_target) >= 5 else 0.5

    X_train, X_test, y_train, y_test = train_test_split(
        X_target,
        y,
        test_size=test_size,
        random_state=42,
        stratify=stratify,
    )

    # Preprocessing and classification pipeline
    model = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        (
            "classifier",
            RandomForestClassifier(
                n_estimators=100,
                random_state=42,
                class_weight="balanced",
                n_jobs=-1,
            ),
        ),
    ])

    model.fit(X_train, y_train)

    # Evaluate model
    predictions = model.predict(X_test)
    accuracy = accuracy_score(y_test, predictions)

    models[target] = model
    label_encoders[target] = encoder

    results[target] = {
        "accuracy": accuracy,
        "training_rows": len(X_train),
        "testing_rows": len(X_test),
        "classes": list(encoder.classes_),
    }

    print(f"\n{'=' * 50}")
    print(f"TARGET: {target}")
    print(f"Accuracy: {accuracy:.2%}")
    print(f"Training rows: {len(X_train)}")
    print(f"Testing rows: {len(X_test)}")
    print(
        classification_report(
            y_test,
            predictions,
            labels=np.arange(len(encoder.classes_)),
            target_names=encoder.classes_,
            zero_division=0,
        )
    )
    print("-" * 50 + "\n")

  # Summary of model performance
  if results:
    results_df = pd.DataFrame(results).T
    display(results_df)
  else:
    print(
        "\nNo models were successfully trained. Annotate and save more rows with"
        " edited = '1' in your UI to build out your dataset!"
    )

C:\Users\nishw\AppData\Local\Temp\ipykernel_26996\3941137672.py:28: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, db)


Successfully loaded 13 filtered rows (edited = '1') from MySQL database.
Valid target columns found: ['was', 'then', 'currently', 'future', 'volatility', 'bias', 'phase', 'momentum_direction', 'momentum_strength', 'setup', 'momentum_candle', 'decision']

Target 'was': found 12 valid labeled rows.
Skipping was: only one unique label found (['Pushing Down']).

Target 'then': found 12 valid labeled rows.

TARGET: then
Accuracy: 66.67%
Training rows: 9
Testing rows: 3
                      precision    recall  f1-score   support

             Push up       0.00      0.00      0.00         1
Pushed Up with a gap       0.67      1.00      0.80         2

            accuracy                           0.67         3
           macro avg       0.33      0.50      0.40         3
        weighted avg       0.44      0.67      0.53         3

--------------------------------------------------

Target 'currently': found 12 valid labeled rows.

TARGET: currently
Accuracy: 100.00%
Training rows: 9
T

,accuracy,training_rows,testing_rows,classes
then,0.666667,9,3,"[Push up, Pushed Up with a gap]"
currently,1.0,9,3,"[again a gap, push up, pushing down, pushing u..."
future,1.0,9,3,"[pull down form a HL to push up, unknown, will..."
volatility,1.0,9,3,"[good, ok, too_much]"
bias,1.0,9,3,"[none, up]"
phase,1.0,9,3,"[none, pull, push]"
momentum_direction,0.666667,9,3,"[bearish, bullish, up]"
momentum_strength,0.666667,9,3,"[none, strong]"
setup,1.0,9,3,"[HL, MDB, none]"
momentum_candle,1.0,9,3,"[no, yes]"


In [21]:
import mysql.connector
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score, classification_report
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder

# ONNX specific imports
from skl2onnx import convert_sklearn
from skl2onnx.common.data_types import FloatTensorType


def get_connection():
  return mysql.connector.connect(
      host="localhost",
      user="root",
      password="",
      database="marketdata",
  )


# ------------------------------------------
# 1. FETCH DATA FROM MYSQL WITH CONSTRAINT
# ------------------------------------------
try:
  db = get_connection()
  query = 'SELECT * FROM `labeled_flatdata` WHERE `edited` = "1"'
  df = pd.read_sql(query, db)
  print(
      f"Successfully loaded {len(df)} filtered rows (edited = '1') from MySQL"
      " database."
  )
except Exception as e:
  print(f"Database connection or query failed: {e}")
  df = pd.DataFrame()
finally:
  if "db" in locals() and db.is_connected():
    db.close()


# ------------------------------------------
# 2. PROCEED ONLY IF DATA IS LOADED
# ------------------------------------------
if df.empty:
  print(
      "DataFrame is empty. Check your database connection, credentials, and"
      " whether any rows have edited = '1'."
  )
else:
  annotation_columns = [
      "was",
      "then",
      "currently",
      "future",
      "volatility",
      "bias",
      "phase",
      "momentum_direction",
      "momentum_strength",
      "setup",
      "momentum_candle",
      "decision",
  ]

  valid_targets = [col for col in annotation_columns if col in df.columns]
  print(f"Valid target columns found: {valid_targets}\n")

  # ------------------------------------------
  # 3. INPUT FEATURES: M1 PRICE COLUMNS
  # ------------------------------------------
  feature_columns = [
      col
      for col in df.columns
      if col.startswith("M1_")
      and not col.endswith("_time")
      and col.rsplit("_", 1)[-1] in {"open", "high", "low", "close"}
  ]

  print(feature_columns)

  X = df[feature_columns].copy()
  X = X.apply(pd.to_numeric, errors="coerce")
  X = X.replace([np.inf, -np.inf], np.nan)

  # ------------------------------------------
  # 4. TRAIN A MODEL FOR EACH TARGET & EXPORT ONNX
  # ------------------------------------------
  models = {}
  label_encoders = {}
  results = {}

  for target in valid_targets:
    y_series = df[target].fillna("").astype(str).str.strip()
    valid = (
        df[target].notna()
        & y_series.ne("")
        & y_series.ne("nan")
        & y_series.ne("None")
        & y_series.ne("NULL")
    )

    X_target = X.loc[valid]
    y_raw = y_series.loc[valid]

    print(f"Target '{target}': found {len(y_raw)} valid labeled rows.")

    if len(y_raw) < 2:
      print(
          f"Skipping {target}: insufficient labeled rows ({len(y_raw)} found,"
          " need at least 2)."
      )
      continue

    encoder = LabelEncoder()
    y = encoder.fit_transform(y_raw)

    if len(encoder.classes_) < 2:
      print(
          f"Skipping {target}: only one unique label found ({encoder.classes_}).\n"
      )
      continue

    class_counts = pd.Series(y).value_counts()
    stratify = y if class_counts.min() >= 2 else None
    test_size = 0.2 if len(X_target) >= 5 else 0.5

    X_train, X_test, y_train, y_test = train_test_split(
        X_target,
        y,
        test_size=test_size,
        random_state=42,
        stratify=stratify,
    )

    model = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        (
            "classifier",
            RandomForestClassifier(
                n_estimators=100,
                random_state=42,
                class_weight="balanced",
                n_jobs=-1,
            ),
        ),
    ])

    model.fit(X_train, y_train)

    predictions = model.predict(X_test)
    accuracy = accuracy_score(y_test, predictions)

    models[target] = model
    label_encoders[target] = encoder

    results[target] = {
        "accuracy": accuracy,
        "training_rows": len(X_train),
        "testing_rows": len(X_test),
        "classes": list(encoder.classes_),
    }

    print(f"\n{'=' * 50}")
    print(f"TARGET: {target}")
    print(f"Accuracy: {accuracy:.2%}")
    print(
        classification_report(
            y_test,
            predictions,
            labels=np.arange(len(encoder.classes_)),
            target_names=encoder.classes_,
            zero_division=0,
        )
    )

    # ------------------------------------------
    # EXPORT MODEL TO ONNX FORMAT
    # ------------------------------------------
    try:
      # Define input type and shape dynamically based on feature count
      initial_type = [("float_input", FloatTensorType([None, X_target.shape[1]]))]

      # Convert model pipeline to ONNX
      onnx_model = convert_sklearn(model, initial_types=initial_type)

      # Save to disk
      onnx_filename = f"model_{target}.onnx"
      with open(onnx_filename, "wb") as f:
        f.write(onnx_model.SerializeToString())
      print(f"Successfully exported {target} model to '{onnx_filename}'")
    except Exception as e:
      print(f"Failed to export {target} to ONNX: {e}")

    print("-" * 50 + "\n")

  if results:
    results_df = pd.DataFrame(results).T
    display(results_df)

C:\Users\nishw\AppData\Local\Temp\ipykernel_26996\1890954838.py:31: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, db)


Successfully loaded 13 filtered rows (edited = '1') from MySQL database.
Valid target columns found: ['was', 'then', 'currently', 'future', 'volatility', 'bias', 'phase', 'momentum_direction', 'momentum_strength', 'setup', 'momentum_candle', 'decision']

['M1_01_open', 'M1_01_high', 'M1_01_low', 'M1_01_close', 'M1_02_open', 'M1_02_high', 'M1_02_low', 'M1_02_close', 'M1_03_open', 'M1_03_high', 'M1_03_low', 'M1_03_close', 'M1_04_open', 'M1_04_high', 'M1_04_low', 'M1_04_close', 'M1_05_open', 'M1_05_high', 'M1_05_low', 'M1_05_close', 'M1_06_open', 'M1_06_high', 'M1_06_low', 'M1_06_close', 'M1_07_open', 'M1_07_high', 'M1_07_low', 'M1_07_close', 'M1_08_open', 'M1_08_high', 'M1_08_low', 'M1_08_close', 'M1_09_open', 'M1_09_high', 'M1_09_low', 'M1_09_close', 'M1_10_open', 'M1_10_high', 'M1_10_low', 'M1_10_close', 'M1_11_open', 'M1_11_high', 'M1_11_low', 'M1_11_close', 'M1_12_open', 'M1_12_high', 'M1_12_low', 'M1_12_close', 'M1_13_open', 'M1_13_high', 'M1_13_low', 'M1_13_close', 'M1_14_open', 'M

,accuracy,training_rows,testing_rows,classes
then,0.666667,9,3,"[Push up, Pushed Up with a gap]"
currently,1.0,9,3,"[again a gap, push up, pushing down, pushing u..."
future,1.0,9,3,"[pull down form a HL to push up, unknown, will..."
volatility,1.0,9,3,"[good, ok, too_much]"
bias,1.0,9,3,"[none, up]"
phase,1.0,9,3,"[none, pull, push]"
momentum_direction,0.666667,9,3,"[bearish, bullish, up]"
momentum_strength,0.666667,9,3,"[none, strong]"
setup,1.0,9,3,"[HL, MDB, none]"
momentum_candle,1.0,9,3,"[no, yes]"


In [11]:

# Check how many rows were loaded
print("Total rows:", len(df))

annotation_columns = [
    "analysis",
    "bias",
    "phase",
    "momentum_direction",
    "momentum_strength",
    "setup",
    "momentum_candle",
    "decision",
]

# Check non-empty labels for each target
for col in annotation_columns:
    if col in df.columns:
        values = df[col].astype("string").str.strip()
        valid = values.notna() & values.ne("")

        print(
            f"{col:25} "
            f"labeled rows: {valid.sum():5} "
            f"unique labels: {values[valid].nunique():5}"
        )
    else:
        print(f"{col:25} COLUMN NOT FOUND")


Total rows: 21
analysis                  labeled rows:     2 unique labels:     1
bias                      labeled rows:     2 unique labels:     1
phase                     labeled rows:     2 unique labels:     1
momentum_direction        labeled rows:     4 unique labels:     4
momentum_strength         labeled rows:     0 unique labels:     0
setup                     labeled rows:     0 unique labels:     0
momentum_candle           labeled rows:     0 unique labels:     0
decision                  labeled rows:     2 unique labels:     1


In [22]:
import json

# (Inside your training loop, right after successful ONNX export):
# Save the label encoder classes so you know what integer corresponds to what string label
classes_filename = f"classes_{target}.json"
with open(classes_filename, "w") as f:
  json.dump(list(encoder.classes_), f)
print(f"Saved label classes to '{classes_filename}'")

Saved label classes to 'classes_decision.json'


In [23]:
import json
import numpy as np
import onnxruntime as ort
import pandas as pd

# 1. Choose which target model you want to load (e.g., 'bias', 'setup', 'future')
target_name = "bias"

onnx_model_path = f"model_{target_name}.onnx"
classes_path = f"classes_{target_name}.json"

# 2. Load the ONNX runtime session
session = ort.InferenceSession(onnx_model_path)

# 3. Load the corresponding label encoder classes
with open(classes_path, "r") as f:
  class_labels = json.load(f)

print(f"Loaded model '{onnx_model_path}' with classes: {class_labels}")

# Get the exact input name expected by the ONNX model
input_name = session.get_inputs()[0].name
print(f"Model expects input node name: '{input_name}'")

# 4. Prepare new input features (e.g., a single row or batch of M1 price data)
# Note: Ensure these features match the exact same columns and order your model was trained on!
# For demonstration, let's simulate a single row of input data matching your feature columns shape:
num_features = session.get_inputs()[0].shape[
    1
]  # Dynamically gets expected feature count
dummy_input_row = np.random.rand(1, num_features).astype(
    np.float32
)  # Replace with your actual M1 data array

# 5. Run Inference
outputs = session.run(None, {input_name: dummy_input_row})

# ONNX scikit-learn classifiers typically return two outputs:
# - outputs[0]: The predicted class label (integer)
# - outputs[1]: Probabilities for each class
predicted_class_idx = outputs[0][0]
class_probabilities = outputs[1]

# 6. Map the integer prediction back to your original string label
predicted_label = class_labels[int(predicted_class_idx)]

print(f"\nPrediction Results for '{target_name}':")
print(f"-> Predicted String Label: {predicted_label}")
print(f"-> Raw Class Probabilities: {class_probabilities}")

FileNotFoundError: [Errno 2] No such file or directory: 'classes_bias.json'